<a href="https://colab.research.google.com/github/marcocslima/rag_tests/blob/main/Llamaindex_H%C3%ADbrido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Instalação

In [ ]:
!pip install -q llama-index-core llama-index-embeddings-huggingface llama-index-retrievers-bm25 llama-index-llms-gemini llama-index-readers-file llama-index
!pip uninstall -y -q transformers tokenizers sentence-transformers
!pip install -q --no-cache-dir "transformers==4.41.2" "tokenizers==0.19.1" "sentence-transformers==2.7.0"
!pip install -q llama-index llama-index-llms-groq         # Para Groq
!pip install -q llama-index llama-index-llms-openrouter  # Para OpenRouter

from IPython.display import clear_output
clear_output()

#Imports + modelos (LLM opcional)

In [ ]:
import os, re
from typing import List, Dict, Optional

from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.schema import TextNode

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import VectorIndexRetriever

# opcional (para responder bonito). Se você só quer testar retrieval, pode ignorar o LLM.
from llama_index.llms.groq import Groq

In [ ]:
import transformers, tokenizers
print("transformers:", transformers.__version__, "tokenizers:", tokenizers.__version__)

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")

clear_output()
print("OK embed loaded")

In [ ]:
from google.colab import userdata

# OPÇÃO A: GROQ
#from llama_index.llms.groq import Groq
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY_MCL')
llm = Groq(model="llama-3.3-70b-versatile") # Modelo potente para extração jurídica

# OPÇÃO B: OPENROUTER
# from llama_index.llms.openrouter import OpenRouter
# os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
# llm = OpenRouter(model="meta-llama/llama-3-70b-instruct") # Exemplo de modelo

#Upload/Leitura do Markdown

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione lei_complementar-460-2008.md
path = next(iter(uploaded.keys()))
md = uploaded[path].decode("utf-8", errors="ignore")
len(md)

#Limpeza do texto

In [ ]:
def clean_markdown(md: str) -> str:
    # remove imagens markdown
    md = re.sub(r'!\[.*?\]\(.*?\)\s*', '', md)

    # remove cabeçalhos/rodapés repetidos comuns nesse arquivo
    md = re.sub(r'(?m)^\s*#\s*C[aâ]mara Municipal de Jundia[ií]\s*$', '', md)
    md = re.sub(r'(?m)^\s*##?\s*Estado de S[aã]o Paulo\s*$', '', md)
    md = re.sub(r'(?m)^\s*\(Texto compilado.*?\)\s*$', '', md)

    # padroniza "Art." removendo negrito em volta
    md = re.sub(r'\*\*\s*(Art\.\s*\d+[ºo]?\.)\s*\*\*', r'\1', md)

    # colapsa espaços excessivos
    md = re.sub(r'\n{3,}', '\n\n', md).strip()
    return md

md_clean = clean_markdown(md)
md_clean[:1000]

#Split “por Artigo” + metadata (a chave do sucesso)

In [ ]:
ART_RE = re.compile(r'(?mi)^\s*Art\.\s*(\d+)\s*[ºo]?\.\s*')

def split_nodes_por_artigo(text: str, lei: str = "LC 460/2008") -> List[TextNode]:
    matches = list(ART_RE.finditer(text))
    nodes: List[TextNode] = []

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        artigo_num = int(m.group(1))
        artigo_text = text[start:end].strip()

        nodes.append(TextNode(
            text=artigo_text,
            metadata={"lei": lei, "artigo": artigo_num}
        ))

    return nodes

nodes = split_nodes_por_artigo(md_clean, lei="LC 460/2008")
len(nodes), nodes[0].metadata, nodes[0].text[:200]

In [ ]:
arts = [n for n in nodes if n.metadata.get("artigo") == 15]
len(arts), (arts[0].text[:400] if arts else "não achou")

#Índices e Retrievers (BM25 + Vetorial)

In [ ]:
# # Vetorial
# v_index = VectorStoreIndex(nodes)
# v_retriever = VectorIndexRetriever(index=v_index, similarity_top_k=10)

# # Lexical (BM25) - perfeito para "Art. 166"
# bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5)

In [ ]:
from tqdm.auto import tqdm
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex

embed_model = Settings.embed_model

texts = [n.get_content(metadata_mode="none") for n in nodes]

batch_size = 16  # CPU: 8 ou 16 costuma ser estável; 32 se aguentar
embs = []

for i in tqdm(range(0, len(texts), batch_size), desc="Gerando embeddings"):
    batch = texts[i:i+batch_size]
    embs.extend(embed_model.get_text_embedding_batch(batch))

for n, e in zip(nodes, embs):
    n.embedding = e

# Agora o índice vetorial fica bem mais rápido (não precisa embeddar de novo)
v_index = VectorStoreIndex(nodes)

In [ ]:
v_retriever = VectorIndexRetriever(index=v_index, similarity_top_k=10)

# Lexical (BM25) - perfeito para "Art. 166"
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5)

#Router de retrieval (Artigo -> BM25)

In [ ]:
def extrair_num_artigo(query: str) -> Optional[int]:
    m = re.search(r'(?i)\bart\.?\s*(\d+)\b|\bartigo\s*(\d+)\b', query)
    if not m:
        return None
    return int(m.group(1) or m.group(2))

def retrieve(query: str):
    art = extrair_num_artigo(query)

    if art is not None:
        # BM25 com string “Art. 166” tende a acertar com muita consistência
        hits = bm25_retriever.retrieve(f"Art. {art}.")
        return hits

    # fallback semântico
    return v_retriever.retrieve(query)

#Debug

In [ ]:
query = "Me traga o texto do Artigo 92 do Decreto 27.250/2017."
hits = retrieve(query)

for h in hits:
    print("score=", round(h.score, 4), "meta=", h.node.metadata)
    print(h.node.text[:250])
    print("----")